# Q-RBnBR：首个论文到 solver 的端到端复现

这本 notebook 复现论文 S1 的小规模随机无权 MaxCut 路线：问题定义来自 `problem/reproductions/qrbnbr.py`，算法实现来自 `lib/solvers/qubo/qrbnbr.py`。Notebook 只负责配置、调用、展示和断言，不承载算法逻辑。

> 这里的 `optimal` 来自 admissible bound 与 edge-parity tree 穷尽；p=1 QAOA/QRR 只提供候选解和分支信息，不单独构成最优性证明。

## 1. 环境与导入

在 VS Code/Jupyter 中选择项目 `.venv` 对应的 Python 3.11 kernel。

In [ ]:
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, QrbnbrQuboSolver
from problem.graph_codec import graph_from_node_link
from problem.reproductions import build_qrbnbr_s1_instance
from tests.oracles import enumerate_qubo, public_json_number

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 生成 S1 风格问题

生成器固定 seed，返回 `qubo.v1`，同时在 metadata 中保留 NetworkX node-link 源图、论文链接、源码 revision 和复现范围。

In [ ]:
problem = build_qrbnbr_s1_instance(
    variable_count=10,
    edge_probability=0.45,
    seed=2025,
)
validate_qubo(problem)

graph = graph_from_node_link(problem["metadata"]["source_graph"])
reproduction = problem["metadata"]["reproduction"]

print("Problem:", problem["problem_id"])
print("Graph nodes / edges:", graph.number_of_nodes(), graph.number_of_edges())
print("Experiment family:", reproduction["experiment_family"])
print("Source revision:", reproduction["source_revision"])

## 3. 独立 oracle 与 Exact 基线

`tests.oracles` 用独立 `Fraction` 穷举获得真实最优值；`ExactQuboSolver` 验证 production solver 契约。

In [ ]:
oracle_rows = enumerate_qubo(problem)
oracle_energy = public_json_number(oracle_rows[0]["energy_exact"])
exact_result = ExactQuboSolver().solve(problem)

validate_qubo_result(problem, exact_result)
assert exact_result["status"] == "optimal"
assert exact_result["best_energy"] == oracle_energy

print("Oracle optimum energy:", oracle_energy)
print("Oracle assignments checked:", len(oracle_rows))
print("Exact sample:", exact_result["best_sample"])

## 4. 运行 Q-RBnBR 复现

分别运行 R1/raw correlation 与 R2/selective composition。两者共享相同的 p=1 QAOA、QRR rounding、Laplacian bound 和 exact leaf closure。

In [ ]:
common_config = {
    "brute_force_threshold": 4,
    "qaoa_grid_size": 5,
    "qaoa_refinement_steps": 4,
    "max_variables": 18,
}

r1_result = QrbnbrQuboSolver().solve(
    problem,
    {**common_config, "branching_rule": "r1"},
)
r2_selective_result = QrbnbrQuboSolver().solve(
    problem,
    {
        **common_config,
        "branching_rule": "r2",
        "branching_matrix": "selective",
        "selective_rank": 3,
    },
)

for result in (r1_result, r2_selective_result):
    validate_qubo_result(problem, result)
    assert result["status"] == "optimal"
    assert result["best_energy"] == oracle_energy
    assert result["metrics"]["tree_exhausted"] is True
    json.dumps(result, allow_nan=False)

print("R1 energy / nodes:", r1_result["best_energy"], r1_result["metrics"]["nodes_explored"])
print("R2 selective energy / nodes:", r2_selective_result["best_energy"], r2_selective_result["metrics"]["nodes_explored"])

## 5. 可审计结果摘要

节点数、prune 数、QRR 子问题数和 exact leaf 数都由 solver 脚本记录。此处只展示同一问题上的路线差异。

In [ ]:
summary = []
for label, result in (
    ("R1 + correlation", r1_result),
    ("R2 + selective", r2_selective_result),
):
    summary.append({
        "route": label,
        "status": result["status"],
        "energy": result["best_energy"],
        "nodes": result["metrics"]["nodes_explored"],
        "pruned": result["metrics"]["nodes_pruned"],
        "qrr_subproblems": result["metrics"]["qrr_subproblems"],
        "exact_leaves": result["metrics"]["exact_subproblems"],
    })

summary

## 结论与边界

- 本案例贯通了论文来源、确定性问题脚本、canonical QUBO、solver 脚本、独立 oracle 和 notebook。
- 当前复现声明仅覆盖非负权 MaxCut QUBO；一般 QUBO 的 signed/anchored MaxCut adapter 是工程扩展，不计入论文忠实复现。
- 当前使用 NumPy 精确 statevector 复现 p=1 QAOA correlation，不引入真实量子硬件噪声或 1024-shot sampling 实验；这应作为后续实验层任务单独加入。